# Cutting Stock Problem

Una fábrica cuenta con un stock (existencia) de rollos estándar de 100 metros de
ancho y pueden procesar órdenes de 14, 31, 36 y 45 metros de ancho. Este mes dicha
fábrica recibió los siguientes pedidos:


| Ancho del pedido $w_i$ | Cantidad del pedido (Demanda $b_i$) |
| ---------------------- | ----------------------------------- |
| 14                     | 211                                 |
| 31                     | 395                                 |
| 36                     | 610                                 |
| 45                     | 97                                  |

Formular un modelo de programación entera que permita a la fábrica decidir cómo
cortar los rollos para cumplir las órdenes demandadas, minimizando los residuos del
proceso de corte.

## Formulación matemática

### Generación de patrones
Un patrón es una combinación de piezas $(n_{45},n_{36},n_{31},n_{14})$ cuya suma de anchos no excede $W=100$. Tomando cada ancho por separado, llevado a su máximo, y rellenando el sobrante con el siguiente ancho que quepa, se obtienen 4 patrones:

| Patrón $j$ | 45 | 36 | 31 | 14 | Desperdicio |
| ---------- | -- | -- | -- | -- | ----------- |
| 1          | 2  | 0  | 0  | 0  | 10          |
| 2          | 0  | 2  | 0  | 2  | 0           |
| 3          | 0  | 0  | 3  | 0  | 7           |
| 4          | 0  | 0  | 0  | 7  | 2           |

$a_{ij}$ = número de piezas de ancho $i$ que produce el patrón $j$ (la tabla anterior, leída por columnas).

*Nota:* este es un subconjunto reducido de patrones, armado a mano llevando cada ancho a su máximo por separado. Un generador exhaustivo (cuatro ciclos anidados sobre los cuatro anchos) produciría muchos más patrones, incluyendo mezclas de dos o más anchos grandes a la vez, y podría dar un óptimo con menos rollos totales que el que se obtiene aquí.

In [1]:
from docplex.mp.model import Model

# patrones: [45, 36, 31, 14]
patrones_ij = [
    [2, 0, 0, 0],  # patron 1
    [0, 2, 0, 2],  # patron 2
    [0, 0, 3, 0],  # patron 3
    [0, 0, 0, 7],  # patron 4
]
demanda_i = [97, 610, 395, 211]  # 45, 36, 31, 14, en ese orden

mdl = Model(name='Cutting Stock P1')

n = 4  # patrones
m = 4  # anchos pedidos

In [2]:
all_patrones_ij = []

for i in range (int(100/45)+1): 
    for j in range(int((100-i*45) / 36) +1): 
        for k in range(int((100-i*45 -j*36) / 31) +1): 
            for l in range(int((100-i*45 -j*36 - k*31) / 14) +1): 
                    if (i*45 + j*36 + k*31 + l*14 <= 100) and (i+j+k+l > 0):
                        all_patrones_ij.append( (i, j, k, l) )

print(len(all_patrones_ij)) # 45, 36, 31, 14

all_patrones_ij # 45, 36, 31, 14

37


[(0, 0, 0, 1),
 (0, 0, 0, 2),
 (0, 0, 0, 3),
 (0, 0, 0, 4),
 (0, 0, 0, 5),
 (0, 0, 0, 6),
 (0, 0, 0, 7),
 (0, 0, 1, 0),
 (0, 0, 1, 1),
 (0, 0, 1, 2),
 (0, 0, 1, 3),
 (0, 0, 1, 4),
 (0, 0, 2, 0),
 (0, 0, 2, 1),
 (0, 0, 2, 2),
 (0, 0, 3, 0),
 (0, 1, 0, 0),
 (0, 1, 0, 1),
 (0, 1, 0, 2),
 (0, 1, 0, 3),
 (0, 1, 0, 4),
 (0, 1, 1, 0),
 (0, 1, 1, 1),
 (0, 1, 1, 2),
 (0, 1, 2, 0),
 (0, 2, 0, 0),
 (0, 2, 0, 1),
 (0, 2, 0, 2),
 (1, 0, 0, 0),
 (1, 0, 0, 1),
 (1, 0, 0, 2),
 (1, 0, 0, 3),
 (1, 0, 1, 0),
 (1, 0, 1, 1),
 (1, 1, 0, 0),
 (1, 1, 0, 1),
 (2, 0, 0, 0)]

### Variables de decisión
$$x_j = \text{número de rollos estándar a cortar de acuerdo al patrón } j,\qquad x_j\ge0 \text{ y entero}$$


In [3]:
# x_j = rollos cortados con el patron j
x = mdl.integer_var_list(n, name='x')

### Restricción de demanda
Se debe cumplir con la demanda de cada ancho $i$:
$$\sum_{j=1}^4 a_{ij}x_j \ge b_i,\quad i=1,\dots,4$$

Con los datos de la tabla:
$$2x_1 \ge 97 \quad(45)$$
$$2x_2 \ge 610 \quad(36)$$
$$3x_3 \ge 395 \quad(31)$$
$$2x_2+7x_4 \ge 211 \quad(14)$$


In [4]:
# Restriccion de demanda
#mdl.add_constraints(
#    mdl.sum(patrones_ij[j][i] * x[j] for j in range(n)) >= demanda_i[i]
#    for i in range(m))

# Restriccion de demanda
mdl.add_constraints(
    mdl.sum(all_patrones_ij[j][i] * x[j] for j in range(n)) >= demanda_i[i]
    for i in range(m))

Error: Adding trivially infeasible linear constraint: '0 >= 97', pos: 1
Error: Adding trivially infeasible linear constraint: '0 >= 610', pos: 1
Error: Adding trivially infeasible linear constraint: '0 >= 395', pos: 1


[docplex.mp.LinearConstraint[](0,GE,97),
 docplex.mp.LinearConstraint[](0,GE,610),
 docplex.mp.LinearConstraint[](0,GE,395),
 docplex.mp.LinearConstraint[](x_0+2x_1+3x_2+4x_3,GE,211)]

### Función objetivo
Minimizar el número de rollos estándar necesarios.
$$\min z = \sum_{j=1}^4 x_j$$

In [5]:
# Minimizar el numero de rollos
mdl.minimize(
    mdl.sum(x[j] for j in range(n))
    )

# Resolver e imprimir solucion
solution = mdl.solve(log_output=True,)


print(solution)

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Infeasibility row 'c1':  0 >= 97.
Presolve time = 0.00 sec. (0.00 ticks)

Root node processing (before b&c):
  Real time             =    0.02 sec. (0.00 ticks)
Parallel b&c, 4 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.02 sec. (0.00 ticks)
None
